# AI Mathematical Olympiad — Full Pipeline

**MCTS-based Math Solver with Symbolic Verification**

This notebook runs the complete pipeline:
1. Install dependencies & verify GPU
2. Data pipeline — stream OpenMathReasoning TIR → parquet
3. SFT training — QLoRA fine-tune NuminaMath-7B-TIR
4. MCTS-RL training — generate verified solutions via MCTS, fine-tune on them
5. Evaluation — compare base vs SFT vs MCTS-RL
6. Publication-quality plots
7. Download trained adapters

**Before running:** Runtime → Change runtime type → **T4 GPU**

---
## 1. Setup & GPU Verification

In [ ]:
!pip install -q transformers>=4.44.0 datasets peft bitsandbytes accelerate trl>=0.9.0 \
    matplotlib sympy pandas pyarrow tqdm

In [ ]:
import torch
import os

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

GPU_NAME  = torch.cuda.get_device_name(0)
VRAM_GB   = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")
print(f"PyTorch: {torch.__version__}")

# Directories
os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/models/sft", exist_ok=True)
os.makedirs("/content/models/mcts_rl", exist_ok=True)
os.makedirs("/content/experiments", exist_ok=True)

---
## 2. Data Pipeline — Stream OpenMathReasoning TIR

Streams from `nvidia/OpenMathReasoning` (TIR split), parses each solution into
structured reasoning steps (text segments, code blocks, outputs, boxed answers),
and writes to a parquet file. Only the current batch is in memory.

**Output schema per row:**
- `problem_id` — unique tracking ID (`tir_000001`, ...)
- `problem` / `solution` / `expected_answer` / `difficulty` — raw fields
- `messages` — chat-format for `tokenizer.apply_chat_template()`
- `parsed_answer` — extracted from `\boxed{}`
- `reasoning_steps` — list of `{step_number, type, content}` dicts
- `num_steps` / `num_code_blocks` / `has_answer` — summary stats

In [ ]:
import gc
import re
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from tqdm.auto import tqdm

DATA_LIMIT = 50_000  # Set to None for full 1.7M rows
BATCH_SIZE = 5_000
OUTPUT_PATH = "/content/data/processed/math_reasoning_tir.parquet"

# --- Structured schema ---
STEP_STRUCT = pa.struct([
    ("step_number", pa.int32()),
    ("type", pa.string()),       # "text", "code", "output", "answer"
    ("content", pa.string()),
])

SCHEMA = pa.schema([
    ("problem_id", pa.string()),
    ("problem", pa.string()),
    ("solution", pa.string()),
    ("expected_answer", pa.string()),
    ("difficulty", pa.string()),
    ("messages", pa.list_(pa.struct([
        ("role", pa.string()),
        ("content", pa.string()),
    ]))),
    ("parsed_answer", pa.string()),
    ("reasoning_steps", pa.list_(STEP_STRUCT)),
    ("num_steps", pa.int32()),
    ("num_code_blocks", pa.int32()),
    ("has_answer", pa.bool_()),
])

_row_counter = 0


# --- Solution parser ---

def parse_solution(solution_text):
    """Parse a TIR solution into structured reasoning steps.

    TIR format alternates between:
      - free text (reasoning/explanation)
      - ```python ... ``` code blocks
      - ```output ... ``` execution results
      - \\boxed{...} final answers

    Returns (steps, parsed_answer) where steps is a list of dicts.
    """
    if not solution_text:
        return [], None

    steps = []
    step_num = 0
    parsed_answer = None

    # Split on code fences while keeping delimiters
    # Pattern captures: ```python\n...\n``` and ```output\n...\n```
    parts = re.split(r'(```(?:python|output)\n.*?```)', solution_text, flags=re.DOTALL)

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part.startswith('```python\n') and part.endswith('```'):
            step_num += 1
            code = part[len('```python\n'):-len('```')].strip()
            steps.append({"step_number": step_num, "type": "code", "content": code})

        elif part.startswith('```output\n') and part.endswith('```'):
            step_num += 1
            output = part[len('```output\n'):-len('```')].strip()
            steps.append({"step_number": step_num, "type": "output", "content": output})

        else:
            # Text segment — may contain \boxed{} answer
            boxed = re.findall(r'\\boxed\{(.+?)\}', part)
            if boxed:
                parsed_answer = boxed[-1]  # last boxed is the final answer

            # Split text on blank lines into logical chunks
            chunks = re.split(r'\n\s*\n', part)
            for chunk in chunks:
                chunk = chunk.strip()
                if not chunk:
                    continue
                step_num += 1
                # Tag chunks that contain a boxed answer
                step_type = "answer" if re.search(r'\\boxed\{', chunk) else "text"
                steps.append({"step_number": step_num, "type": step_type, "content": chunk})

    return steps, parsed_answer


def build_tir_messages(problem, solution):
    return [
        {"role": "system", "content": ""},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": solution},
    ]


def format_tir_chat(problem, solution):
    return (
        f"<|system|>\n<|end|>\n"
        f"<|user|>\n{problem}<|end|>\n"
        f"<|assistant|>\n{solution}<|end|>"
    )


def transform_row(example):
    global _row_counter
    _row_counter += 1
    problem_id = f"tir_{_row_counter:06d}"
    problem = example["problem"]
    solution = example["generated_solution"]

    steps, parsed_answer = parse_solution(solution)
    num_code = sum(1 for s in steps if s["type"] == "code")

    return {
        "problem_id": problem_id,
        "problem": problem,
        "solution": solution,
        "expected_answer": example["expected_answer"],
        "difficulty": example.get("problem_source", "unknown"),
        "messages": build_tir_messages(problem, solution),
        "parsed_answer": parsed_answer,
        "reasoning_steps": steps,
        "num_steps": len(steps),
        "num_code_blocks": num_code,
        "has_answer": parsed_answer is not None,
    }


print("Streaming nvidia/OpenMathReasoning (tir split)...")
ds = load_dataset("nvidia/OpenMathReasoning", split="tir", streaming=True)

writer = None
batch = []
rows_written = 0

for example in tqdm(ds, desc="Processing", total=DATA_LIMIT):
    if DATA_LIMIT and rows_written + len(batch) >= DATA_LIMIT:
        break
    batch.append(transform_row(example))

    if len(batch) >= BATCH_SIZE:
        table = pa.Table.from_pylist(batch, schema=SCHEMA)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
        writer.write_table(table)
        rows_written += len(batch)
        batch.clear()
        del table
        gc.collect()
        print(f"  Written {rows_written:,} rows")

if batch:
    table = pa.Table.from_pylist(batch, schema=SCHEMA)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
    writer.write_table(table)
    rows_written += len(batch)
    batch.clear()
    del table

if writer:
    writer.close()

del ds, writer
gc.collect()

final_count = pq.read_metadata(OUTPUT_PATH).num_rows
print(f"\nDone: {final_count:,} rows -> {OUTPUT_PATH}")

In [ ]:
# Quick peek at the structured data
import pandas as pd

df_peek = pd.read_parquet(OUTPUT_PATH)
print(f"Total rows: {len(df_peek):,}")
print(f"Has parsed answer: {df_peek['has_answer'].sum():,} / {len(df_peek):,} ({df_peek['has_answer'].mean():.1%})")
print(f"Avg steps per solution: {df_peek['num_steps'].mean():.1f}")
print(f"Avg code blocks per solution: {df_peek['num_code_blocks'].mean():.1f}")

print(f"\nDifficulty distribution:")
print(df_peek["difficulty"].value_counts().head(10))

# Show one parsed example
row = df_peek.iloc[0]
print(f"\n{'='*60}")
print(f"problem_id: {row['problem_id']}")
print(f"Problem: {row['problem'][:200]}...")
print(f"Expected: {row['expected_answer']}")
print(f"Parsed answer: {row['parsed_answer']}")
print(f"Steps ({row['num_steps']} total, {row['num_code_blocks']} code blocks):")
for step in row["reasoning_steps"][:8]:
    preview = step["content"][:100].replace("\n", " ")
    print(f"  [{step['step_number']}] ({step['type']}) {preview}...")

del df_peek
gc.collect()